# 🔬 G-Eval Panel — Penilaian Makalah BPOM

**3 Model Panel** via OpenRouter:
- Model A: `openai/gpt-4o-mini`
- Model B: `anthropic/claude-haiku-4`
- Model C: `google/gemini-flash-1.5`

**Output:**
- Score per kriteria (skala 40–100)
- Confidence score (konsensus 3 model)
- CoT reasoning dari masing-masing model

In [ ]:
# Cell 1: Install dependencies
# !pip install deepeval openai python-dotenv pandas -q

In [ ]:
# Cell 2: Imports & env
import os, json, asyncio, statistics, datetime
from dotenv import load_dotenv

load_dotenv(dotenv_path='../../.env', override=True)

GEVAL_API_KEY  = os.getenv('LLM_BINDING_API_KEY')
GEVAL_BASE_URL = os.getenv('LLM_BINDING_HOST', 'https://openrouter.ai/api/v1')

PANEL_MODELS = {
    'gpt-4o-mini':  os.getenv('GEVAL_MODEL_A'),
    'claude-haiku': os.getenv('GEVAL_MODEL_B'),
    'gemini-flash': os.getenv('GEVAL_MODEL_C'),
}

SCORE_MIN, SCORE_MAX = 40, 100
print('Models:', PANEL_MODELS)
print('API Key set:', bool(GEVAL_API_KEY))

In [ ]:
# Cell 3: Custom DeepEval wrapper untuk OpenRouter
from deepeval.models.base_model import DeepEvalBaseLLM

class OpenRouterGEvalModel(DeepEvalBaseLLM):
    def __init__(self, model_name, alias, api_key, base_url):
        self.model_name = model_name
        self.alias      = alias
        self.api_key    = api_key
        self.base_url   = base_url

    def get_model_name(self): return self.alias
    def load_model(self): return None

    def generate(self, prompt: str, schema=None) -> str:
        from openai import OpenAI
        client = OpenAI(api_key=self.api_key, base_url=self.base_url)
        resp = client.chat.completions.create(
            model=self.model_name,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0, max_tokens=2048,
        )
        return resp.choices[0].message.content

    async def a_generate(self, prompt: str, schema=None) -> str:
        from openai import AsyncOpenAI
        client = AsyncOpenAI(api_key=self.api_key, base_url=self.base_url)
        resp = await client.chat.completions.create(
            model=self.model_name,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0, max_tokens=2048,
        )
        return resp.choices[0].message.content

panel_judges = {
    alias: OpenRouterGEvalModel(model_name, alias, GEVAL_API_KEY, GEVAL_BASE_URL)
    for alias, model_name in PANEL_MODELS.items()
}
print('Judges ready:', list(panel_judges.keys()))

In [ ]:
# Cell 4: Definisi 5 Kriteria G-Eval
GEVAL_CRITERIA_DEF = {
    'n1_kesesuaian_judul': {
        'label': 'Kesesuaian Judul dengan Tema',
        'criteria': 'Nilai sejauh mana judul makalah mencerminkan tema yang ditetapkan dalam ketentuan penulisan makalah. Judul harus spesifik, relevan, dan mencerminkan isu strategis sesuai konteks jabatan.',
        'steps': [
            'Baca judul makalah secara seksama.',
            'Identifikasi tema yang ditetapkan dari bagian INPUT (konteks jabatan & tema).',
            'Periksa apakah kata kunci utama tema muncul atau tercermin dalam judul.',
            'Nilai apakah judul cukup spesifik (bukan terlalu umum/luas).',
            'Tentukan skor: sangat sesuai (tinggi), cukup sesuai (sedang), tidak sesuai (rendah).',
        ],
    },
    'n2_kesesuaian_isi': {
        'label': 'Kesesuaian Isi dengan Judul & Tema',
        'criteria': 'Nilai sejauh mana isi makalah konsisten dengan judul dan tema yang ditetapkan. Seluruh pembahasan harus relevan dan tidak menyimpang dari fokus utama.',
        'steps': [
            'Identifikasi janji yang dibuat oleh judul makalah.',
            'Baca isi makalah dan identifikasi topik-topik yang dibahas.',
            'Periksa apakah setiap bagian isi relevan dengan judul dan tema.',
            'Identifikasi apakah ada penyimpangan atau pembahasan yang tidak relevan.',
            'Nilai konsistensi dan koherensi antara judul, tema, dan isi.',
        ],
    },
    'n3_sistematika': {
        'label': 'Sistematika Penulisan',
        'criteria': 'Nilai kelengkapan dan ketepatan struktur penulisan: pendahuluan, tinjauan pustaka, pembahasan, kesimpulan, dan daftar pustaka.',
        'steps': [
            'Identifikasi bagian-bagian struktural yang ada dalam makalah.',
            'Periksa apakah pendahuluan mencakup latar belakang dan tujuan.',
            'Periksa apakah ada landasan teori atau tinjauan pustaka yang memadai.',
            'Periksa kelengkapan bagian pembahasan dan kesimpulan.',
            'Nilai alur logis antar bagian dan ketepatan penggunaan sub-judul.',
        ],
    },
    'n4_ketajaman_analisis': {
        'label': 'Ketajaman Analisis',
        'criteria': 'Nilai kedalaman analisis. Penulis harus menunjukkan pemahaman mendalam, didukung data/fakta, dan dikaitkan dengan konteks BPOM/kebijakan publik.',
        'steps': [
            'Identifikasi permasalahan utama yang dianalisis.',
            'Periksa apakah analisis didukung data, fakta, atau referensi yang relevan.',
            'Nilai apakah penulis menunjukkan pemikiran kritis (bukan deskriptif semata).',
            'Periksa apakah ada kaitan dengan isu kebijakan atau konteks jabatan BPOM.',
            'Nilai kedalaman: apakah analisis menjawab mengapa dan bagaimana.',
        ],
    },
    'n5_penggunaan_bahasa': {
        'label': 'Penggunaan Bahasa',
        'criteria': 'Nilai kualitas bahasa: keformalan, ketepatan tata bahasa, kejelasan ekspresi, konsistensi istilah, dan keterbacaan.',
        'steps': [
            'Periksa apakah bahasa yang digunakan formal dan akademis.',
            'Identifikasi adanya kesalahan tata bahasa atau ejaan yang signifikan.',
            'Nilai kejelasan kalimat: apakah mudah dipahami atau ambigu.',
            'Periksa konsistensi penggunaan istilah teknis.',
            'Nilai secara keseluruhan apakah bahasa mendukung penyampaian ide.',
        ],
    },
}
print('Kriteria:', list(GEVAL_CRITERIA_DEF.keys()))

In [ ]:
# Cell 5: Helper functions
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

def build_metric(crit_key, judge):
    d = GEVAL_CRITERIA_DEF[crit_key]
    return GEval(
        name=d['label'], criteria=d['criteria'],
        evaluation_steps=d['steps'],
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=judge, threshold=0.0,
    )

def scale_score(s): return round(SCORE_MIN + s * (SCORE_MAX - SCORE_MIN), 1)

def compute_confidence(scores):
    if len(scores) < 2: return 1.0
    return round(max(0.0, 1.0 - statistics.stdev(scores) / 0.5), 3)

WEIGHTS = {'n1_kesesuaian_judul':1,'n2_kesesuaian_isi':1,'n3_sistematika':1,
           'n4_ketajaman_analisis':2,'n5_penggunaan_bahasa':1}

print('Helpers loaded.')

In [ ]:
# Cell 6: Fungsi Utama — run_geval_panel()
import time

def run_geval_panel(makalah_text, jabatan, tema_text, assessment_context=''):
    start = time.time()
    input_ctx = f'Jabatan: {jabatan}\n\nTema:\n{tema_text[:2000]}\n\nKonteks Jabatan:\n{assessment_context[:1500]}'
    actual_out = makalah_text[:8000]
    criteria_results = {}

    for crit_key, defn in GEVAL_CRITERIA_DEF.items():
        print(f'\n  Evaluating: {defn["label"]}')
        per_model = {}
        for alias, judge in panel_judges.items():
            print(f'    [{alias}]...', end=' ', flush=True)
            try:
                metric = build_metric(crit_key, judge)
                metric.measure(LLMTestCase(input=input_ctx, actual_output=actual_out))
                s = metric.score or 0.0
                per_model[alias] = {'score_0_1': round(s,3), 'score_scaled': scale_score(s), 'reason': metric.reason or ''}
                print(f'✓ {scale_score(s)}')
            except Exception as e:
                print(f'✗ {e}')
                per_model[alias] = {'score_0_1': 0.0, 'score_scaled': SCORE_MIN, 'reason': str(e)}

        scores_0_1 = [v['score_0_1'] for v in per_model.values()]
        mean_0_1   = statistics.mean(scores_0_1)
        closest    = min(per_model, key=lambda a: abs(per_model[a]['score_0_1'] - mean_0_1))
        criteria_results[crit_key] = {
            'label':             defn['label'],
            'mean_score_scaled': round(statistics.mean([v['score_scaled'] for v in per_model.values()]), 1),
            'confidence':        compute_confidence(scores_0_1),
            'model_scores':      {a: v['score_scaled'] for a,v in per_model.items()},
            'model_reasons':     {a: v['reason'] for a,v in per_model.items()},
            'consensus_reason':  per_model[closest]['reason'],
            'per_model':         per_model,
        }

    total_w      = sum(WEIGHTS.values())
    final_score  = round(sum(criteria_results[k]['mean_score_scaled']*WEIGHTS[k] for k in WEIGHTS) / total_w, 1)
    confidence   = round(statistics.mean(v['confidence'] for v in criteria_results.values()), 3)

    return {
        'jabatan':           jabatan,
        'geval_final_score': final_score,
        'geval_confidence':  confidence,
        'panel_models':      list(PANEL_MODELS.keys()),
        'duration_sec':      round(time.time()-start, 1),
        'criteria':          criteria_results,
    }

print('run_geval_panel() defined.')

In [ ]:
# Cell 7: Input — Ganti dengan makalah & jabatan yang ingin dievaluasi
# ─────────────────────────────────────────────────────────────────────
# Opsi A: Teks langsung
MAKALAH_TEXT = """
JUDUL: Strategi Penguatan Pengawasan Obat dan Makanan

BAB I PENDAHULUAN
BPOM memiliki peran strategis dalam melindungi masyarakat dari risiko obat dan makanan
yang tidak memenuhi syarat. Dalam Renstra 2025-2029, penguatan pengawasan menjadi prioritas.

BAB II TINJAUAN PUSTAKA
Pengawasan diatur dalam UU No. 18 Tahun 2012 dan Peraturan BPOM Nomor 2 Tahun 2024.

BAB III PEMBAHASAN
Tantangan: peredaran produk ilegal, keterbatasan SDM, perkembangan teknologi.
Strategi: digitalisasi sistem berbasis risiko, penguatan kapasitas Balai, kolaborasi lintas K/L.

BAB IV KESIMPULAN
Penguatan pengawasan memerlukan pendekatan komprehensif: teknologi, regulasi, dan SDM kompeten.
"""

JABATAN_TEXT  = "Kepala Biro SDM"
TEMA_TEXT     = "Strategi Penguatan Kelembagaan BPOM dalam Mendukung Renstra 2025-2029"
CONTEXT_TEXT  = "Kepala Biro SDM bertanggung jawab atas pengelolaan SDM BPOM secara menyeluruh."

# Opsi B: Baca dari file
# with open('../data/makalah_sample.txt', encoding='utf-8') as f:
#     MAKALAH_TEXT = f.read()

print(f'Makalah: {len(MAKALAH_TEXT)} karakter | Jabatan: {JABATAN_TEXT}')

In [ ]:
# Cell 8: Jalankan G-Eval Panel
print('=' * 60)
print('MEMULAI G-EVAL PANEL — 3 MODEL JUDGE')
print('=' * 60)

results = run_geval_panel(
    makalah_text=MAKALAH_TEXT,
    jabatan=JABATAN_TEXT,
    tema_text=TEMA_TEXT,
    assessment_context=CONTEXT_TEXT,
)

print(f"\n{'='*60}")
print(f"G-Eval Final Score : {results['geval_final_score']}")
print(f"Overall Confidence : {results['geval_confidence']:.1%}")
print(f"Duration           : {results['duration_sec']}s")
print(f"{'='*60}")

In [ ]:
# Cell 9: Tabel Perbandingan Panel
import pandas as pd

rows = []
for k, v in results['criteria'].items():
    row = {'Kriteria': v['label'], 'Confidence': f"{v['confidence']:.0%}"}
    for alias in results['panel_models']:
        row[alias] = v['model_scores'].get(alias, '-')
    row['Mean'] = v['mean_score_scaled']
    rows.append(row)

# Tambah baris final score
final_row = {'Kriteria': '─── FINAL SCORE ───', 'Confidence': f"{results['geval_confidence']:.0%}"}
for alias in results['panel_models']: final_row[alias] = '-'
final_row['Mean'] = results['geval_final_score']
rows.append(final_row)

df = pd.DataFrame(rows)
print('\n📊 PANEL COMPARISON TABLE')
df

In [ ]:
# Cell 10: CoT Reasoning per Kriteria
print('\n📝 REASONING PER KRITERIA\n' + '-'*60)
for k, v in results['criteria'].items():
    conf_emoji = '🟢' if v['confidence'] >= 0.8 else ('🟡' if v['confidence'] >= 0.6 else '🔴')
    print(f"\n{conf_emoji} {v['label']}")
    print(f"   Score: {v['mean_score_scaled']} | Confidence: {v['confidence']:.0%}")
    print(f"   Skor per model: {v['model_scores']}")
    print(f"   Consensus: {v['consensus_reason'][:400]}")

In [ ]:
# Cell 11: Simpan hasil ke JSON
import datetime
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
os.makedirs('./output', exist_ok=True)
out_path = f'./output/geval_{ts}.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f'✅ Saved: {out_path}')